# InfoGAN + OpenMax (O-S Model) — NIDS Training

Implementation of the model from:
> "Unknown intrusion traffic detection method based on unsupervised learning and open-set recognition"
> (Fang & Xie, 2025, Nature Scientific Reports)

**Pipeline:**
1. Load CICIDS2017 data → remove irrelevant/zero features → 69 features
2. MinMax normalize → zero-pad to 121 → reshape to 11×11×1
3. Train InfoGAN (unsupervised) to learn cluster representations
4. Assign pseudo-labels using classifier Q
5. Fit OpenMax on activation vectors (Weibull tail fitting)
6. Evaluate O-S model: OpenMax detects unknown, SoftMax classifies known

In [2]:
import sys, os

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

NIDS_ROOT = "/content/drive/MyDrive/Colab Notebooks/nids"
sys.path.insert(0, NIDS_ROOT)

import numpy as np
import pandas as pd
import json
import joblib
import matplotlib.pyplot as plt

import tensorflow as tf
import keras

import nids_utils as nu
import infogan_model as ig
import openmax as om

print(f"TF version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TF version: 2.19.0
GPU available: []


## 1. Load & Preprocess Data

The paper trains InfoGAN on **all known traffic types** (not just benign).
We use all CICIDS2017 CSVs and map labels to the 7-class scheme.

In [3]:
# All CSV files (Monday benign + Tuesday–Friday attack files)
ALL_CSV_PATHS = [nu.MONDAY_CSV] + nu.ATTACK_FILES

print(f"Loading {len(ALL_CSV_PATHS)} CSV files...")
for p in ALL_CSV_PATHS:
    print(f"  {os.path.basename(p)}")

Loading 8 CSV files...
  Monday-WorkingHours.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [4]:
# Load, clean, scale, and reshape to 11x11x1 images
images, labels_int, scaler, feature_cols = ig.prepare_infogan_data(ALL_CSV_PATHS)

print(f"\nTotal samples: {len(images)}")
print(f"Features: {len(feature_cols)}")
print(f"Image shape: {images.shape}")
print(f"\nLabel distribution:")
unique, counts = np.unique(labels_int, return_counts=True)
for u, c in zip(unique, counts):
    name = ig.CLASS_NAMES[u] if 0 <= u < len(ig.CLASS_NAMES) else "Unknown"
    print(f"  {u} ({name}): {c:,}")

Combined dataset: 2830743 samples, 67 features
Image shape: (2830743, 11, 11, 1)

Total samples: 2830743
Features: 67
Image shape: (2830743, 11, 11, 1)

Label distribution:
  -1 (Unknown): 2,180
  0 (Normal): 2,273,097
  1 (Botnet): 1,966
  2 (Brute Force): 13,835
  3 (DoS): 380,699
  4 (Infiltration): 36
  5 (PortScan): 158,930


In [5]:
# Save the scaler for later use
SCALER_SAVE = os.path.join(nu.PREPROCESSING_DIR, "scaler_infogan.pkl")
os.makedirs(os.path.dirname(SCALER_SAVE), exist_ok=True)
joblib.dump(scaler, SCALER_SAVE)
print(f"Scaler saved → {SCALER_SAVE}")

# Save feature columns for reproducibility
FEATURE_COLS_SAVE = os.path.join(nu.PREPROCESSING_DIR, "infogan_feature_cols.json")
with open(FEATURE_COLS_SAVE, "w") as f:
    json.dump(feature_cols, f)
print(f"Feature cols saved → {FEATURE_COLS_SAVE}")

Scaler saved → /content/drive/MyDrive/Colab Notebooks/nids/preprocessing_tools/scaler_infogan.pkl
Feature cols saved → /content/drive/MyDrive/Colab Notebooks/nids/preprocessing_tools/infogan_feature_cols.json


## 2. Train/Test Split

Split data into training (known classes) and test sets.
For open-set evaluation, we can hold out some classes as "unknown".

In [6]:
from sklearn.model_selection import train_test_split

# --- Configuration ---
# Known classes for training (paper uses 7 classes as known)
# To test open-set recognition, you can hold out 1-2 classes as "unknown"
KNOWN_CLASSES = [0, 1, 2, 3, 5, 6]  # Hold out class 4 (Infiltration) as unknown
UNKNOWN_CLASSES = [4]                 # Infiltration = simulated zero-day

# Alternatively, use all classes as known (set UNKNOWN_CLASSES = []):
# KNOWN_CLASSES = list(range(ig.NUM_CLASSES))
# UNKNOWN_CLASSES = []

NUM_KNOWN = len(KNOWN_CLASSES)
print(f"Known classes ({NUM_KNOWN}): {[ig.CLASS_NAMES[c] for c in KNOWN_CLASSES]}")
print(f"Unknown classes: {[ig.CLASS_NAMES[c] for c in UNKNOWN_CLASSES]}")

Known classes (6): ['Normal', 'Botnet', 'Brute Force', 'DoS', 'PortScan', 'Web Attack']
Unknown classes: ['Infiltration']


In [7]:
# Filter to known classes for training
known_mask = np.isin(labels_int, KNOWN_CLASSES)
X_known = images[known_mask]
y_known = labels_int[known_mask]

# Remap labels to contiguous 0..NUM_KNOWN-1
label_remap = {old: new for new, old in enumerate(KNOWN_CLASSES)}
y_known_remapped = np.array([label_remap[l] for l in y_known])

# Train/test split on known data
X_train, X_test_known, y_train, y_test_known = train_test_split(
    X_known, y_known_remapped, test_size=0.2, random_state=42, stratify=y_known_remapped
)

# Unknown test data
if UNKNOWN_CLASSES:
    unknown_mask = np.isin(labels_int, UNKNOWN_CLASSES)
    X_test_unknown = images[unknown_mask]
    y_test_unknown_orig = labels_int[unknown_mask]
else:
    X_test_unknown = np.empty((0, 11, 11, 1), dtype=np.float32)
    y_test_unknown_orig = np.array([], dtype=int)

print(f"Training: {X_train.shape[0]:,} samples")
print(f"Test (known): {X_test_known.shape[0]:,} samples")
print(f"Test (unknown): {X_test_unknown.shape[0]:,} samples")

Training: 2,262,821 samples
Test (known): 565,706 samples
Test (unknown): 36 samples


## 3. Build InfoGAN

Architecture from Fig. 6 of the paper:
- Shared Conv2D backbone (3 layers)
- Discriminator head: Dense(1) → Sigmoid
- Classifier Q head: Dense(128) → BN → LeakyReLU → Dense(c) → Softmax
- Generator: Dense → Reshape → 4× Conv2DTranspose → 11×11×1

In [8]:
# Hyperparameters (from paper)
NOISE_DIM = 100
LR = 0.0002
LAMBDA_INFO = 1.0
BATCH_SIZE = 256
EPOCHS = 200

# Build models
backbone = ig.build_shared_backbone()
discriminator = ig.build_discriminator(backbone)
classifier = ig.build_classifier(backbone, num_classes=NUM_KNOWN)
generator = ig.build_generator(NOISE_DIM, num_classes=NUM_KNOWN)

print("=== Backbone ===")
backbone.summary()
print("\n=== Discriminator ===")
discriminator.summary()
print("\n=== Classifier Q ===")
classifier.summary()
print("\n=== Generator ===")
generator.summary()

=== Backbone ===


Model: "shared_backbone"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 11, 11, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 6, 6, 64)       │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 3, 3, 128)      │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 256)      │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 656,832 (2.51 MB)

 Trainable params: 656,832 (2.51 MB)

...

Model: "generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ noise_z             │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_c            │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 106)       │          0 │ noise_z[0][0],    │
│ (Concatenate)       │                   │            │ latent_c[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8192)      │    876,544 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8192)      │     32,768 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 8192)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 4, 4, 512) │          0 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 7, 7, 256) │  2,097,408 │ reshape[0][0]     │
│ (Conv2DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 7, 256) │      1,024 │ conv2d_transpose… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 7, 7, 256) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 9, 9, 128) │    295,040 │ re_lu_1[0][0]     │
│ (Conv2DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 9, 9, 128) │        512 │ conv2d_transpose… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 9, 9, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_2  │ (None, 11, 11,    │     73,792 │ re_lu_2[0][0]     │
│ (Conv2DTranspose)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 11, 11,    │        256 │ conv2d_transpose… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 11, 11,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_3  │ (None, 11, 11, 1) │         65 │ re_lu_3[0][0]     │
│ (Conv2DTranspose)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,377,409 (12.88 MB)

 Trainable params: 3,360,129 (12.82 MB)

 Non-trainable params: 17,280 (67.50 KB)

## 4. Train InfoGAN

In [ ]:
trainer = ig.InfoGANTrainer(
    generator=generator,
    discriminator=discriminator,
    classifier=classifier,
    backbone=backbone,
    noise_dim=NOISE_DIM,
    num_classes=NUM_KNOWN,
    lr=LR,
    lambda_info=LAMBDA_INFO,
)

history = trainer.fit(X_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

In [ ]:
# Plot training losses
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["d_loss"])
axes[0].set_title("Discriminator Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history["g_loss"])
axes[1].set_title("Generator Loss")
axes[1].set_xlabel("Epoch")

axes[2].plot(history["q_loss"])
axes[2].set_title("Classifier Q Loss (Mutual Info)")
axes[2].set_xlabel("Epoch")

fig.suptitle("InfoGAN Training Losses")
fig.tight_layout()
plt.show()

## 5. Classifier Q — Pseudo-label Assignment

After training, the classifier Q can assign class labels to traffic samples.
We evaluate how well Q's predictions match the true labels.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Predict on training data with classifier Q (SoftMax predictions)
y_pred_train = np.argmax(classifier.predict(X_train, verbose=0), axis=1)
y_pred_test = np.argmax(classifier.predict(X_test_known, verbose=0), axis=1)

# Since InfoGAN is unsupervised, Q's class assignments may not align with true labels.
# We use a Hungarian matching to find the best label permutation.
from scipy.optimize import linear_sum_assignment

def find_best_label_mapping(y_true, y_pred, num_classes):
    """Find optimal mapping from predicted clusters to true labels via Hungarian algorithm."""
    cost_matrix = np.zeros((num_classes, num_classes))
    for true_c in range(num_classes):
        for pred_c in range(num_classes):
            # Cost = number of samples where true==true_c but pred!=pred_c
            mask = y_true == true_c
            cost_matrix[true_c, pred_c] = -np.sum(y_pred[mask] == pred_c)
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    # mapping: predicted_label → true_label
    mapping = {col_ind[i]: row_ind[i] for i in range(num_classes)}
    return mapping

mapping = find_best_label_mapping(y_train, y_pred_train, NUM_KNOWN)
print("Best label mapping (predicted → true):")
for pred_c, true_c in sorted(mapping.items()):
    true_name = ig.CLASS_NAMES[KNOWN_CLASSES[true_c]]
    print(f"  Cluster {pred_c} → Class {true_c} ({true_name})")

In [ ]:
# Apply mapping and evaluate
y_pred_train_mapped = np.array([mapping[p] for p in y_pred_train])
y_pred_test_mapped = np.array([mapping[p] for p in y_pred_test])

known_class_names = [ig.CLASS_NAMES[c] for c in KNOWN_CLASSES]

print("=== Training Set ===")
print(f"Accuracy: {accuracy_score(y_train, y_pred_train_mapped):.4f}")

print("\n=== Test Set (Known Classes) ===")
print(f"Accuracy: {accuracy_score(y_test_known, y_pred_test_mapped):.4f}")
print(classification_report(y_test_known, y_pred_test_mapped, target_names=known_class_names))

In [ ]:
# Confusion matrix for test set
cm = confusion_matrix(y_test_known, y_pred_test_mapped)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Confusion Matrix — Classifier Q (Known Classes)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
tick_marks = np.arange(NUM_KNOWN)
ax.set_xticks(tick_marks)
ax.set_xticklabels(known_class_names, rotation=45, ha="right")
ax.set_yticks(tick_marks)
ax.set_yticklabels(known_class_names)
fig.colorbar(im)
fig.tight_layout()
plt.show()

## 6. Fit OpenMax (Algorithm 2: FitHigh)

Extract activation vectors from classifier Q's penultimate layer,
compute Mean Activation Vectors (MAVs), and fit Weibull distributions
on the tail distances.

In [ ]:
# Get activation vectors (128-dim penultimate layer output) for training data
avs_train = trainer.get_activation_vectors(X_train)
print(f"Activation vectors shape: {avs_train.shape}")

# Use the mapped labels (corrected cluster assignments)
# We need to re-map the Q predictions through our Hungarian mapping
# to get consistent labels for MAV computation
y_for_openmax = y_pred_train_mapped  # Use Q's predictions (mapped to true labels)

# Compute MAVs
mavs, class_avs = om.compute_mavs(avs_train, y_for_openmax, NUM_KNOWN)
print(f"MAVs computed for {len(mavs)} classes")
for cls, mav in mavs.items():
    print(f"  Class {cls} ({known_class_names[cls]}): {len(class_avs[cls])} samples, MAV norm={np.linalg.norm(mav):.4f}")

In [ ]:
# Compute distances and fit Weibull
TAIL_SIZE = 20       # eta in the paper (number of tail distances)
ALPHA_RANK = 3       # alpha (number of top classes to recalibrate)
DISTANCE_TYPE = "cosine"  # Paper uses cosine distance

distances = om.compute_distances(class_avs, mavs, distance_type=DISTANCE_TYPE)
weibull_params = om.fit_weibull(distances, tail_size=TAIL_SIZE)

print("\nWeibull parameters per class:")
for cls, params in weibull_params.items():
    if params:
        shape, loc, scale = params
        print(f"  Class {cls} ({known_class_names[cls]}): shape={shape:.4f}, loc={loc:.4f}, scale={scale:.6f}")
    else:
        print(f"  Class {cls} ({known_class_names[cls]}): FAILED to fit")

## 7. Evaluate O-S Model (OpenMax + SoftMax)

**O-S Model decision rule:**
- Run sample through classifier Q → get activation vector
- Apply OpenMax recalibration → get (NUM_KNOWN+1) probabilities
- If argmax = NUM_KNOWN (unknown class) → predict "Unknown"
- Otherwise → use SoftMax classification (known class prediction)

In [ ]:
# Get activation vectors for test data
avs_test_known = trainer.get_activation_vectors(X_test_known)

# Apply OpenMax
preds_known, probs_known = om.openmax_predict(
    avs_test_known, mavs, weibull_params,
    alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
    distance_type=DISTANCE_TYPE,
)

# For known test data, OpenMax should predict known classes (0..NUM_KNOWN-1)
# Prediction == NUM_KNOWN means "unknown"
known_correct = np.sum((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_as_unknown = np.sum(preds_known == NUM_KNOWN)
known_total = len(y_test_known)

print(f"=== Known Class Test Data ({known_total:,} samples) ===")
print(f"Correctly classified: {known_correct:,} ({known_correct/known_total:.4f})")
print(f"Misclassified as unknown: {known_as_unknown:,} ({known_as_unknown/known_total:.4f})")
print(f"Misclassified as wrong known: {known_total - known_correct - known_as_unknown:,}")

In [ ]:
if len(X_test_unknown) > 0:
    # Get activation vectors for unknown test data
    avs_test_unknown = trainer.get_activation_vectors(X_test_unknown)

    # Apply OpenMax
    preds_unknown, probs_unknown = om.openmax_predict(
        avs_test_unknown, mavs, weibull_params,
        alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
        distance_type=DISTANCE_TYPE,
    )

    # For unknown data, OpenMax should predict NUM_KNOWN (unknown class)
    unknown_detected = np.sum(preds_unknown == NUM_KNOWN)
    unknown_total = len(preds_unknown)

    print(f"=== Unknown Class Test Data ({unknown_total:,} samples) ===")
    print(f"Detected as unknown: {unknown_detected:,} ({unknown_detected/unknown_total:.4f})")
    print(f"Misclassified as known: {unknown_total - unknown_detected:,} ({(unknown_total - unknown_detected)/unknown_total:.4f})")

    # Distribution of misclassifications
    misclassified = preds_unknown[preds_unknown != NUM_KNOWN]
    if len(misclassified) > 0:
        print(f"\nMisclassification distribution:")
        for cls in range(NUM_KNOWN):
            cnt = np.sum(misclassified == cls)
            if cnt > 0:
                print(f"  → Class {cls} ({known_class_names[cls]}): {cnt}")
else:
    print("No unknown test data (all classes used as known).")

In [ ]:
# Combined evaluation: known + unknown test data
if len(X_test_unknown) > 0:
    # Build combined true labels:
    # known classes → their remapped label, unknown → NUM_KNOWN
    y_true_combined = np.concatenate([
        y_test_known,
        np.full(len(X_test_unknown), NUM_KNOWN, dtype=int),
    ])
    y_pred_combined = np.concatenate([preds_known, preds_unknown])

    all_class_names = known_class_names + ["Unknown"]
    print("=== Combined O-S Model Evaluation ===")
    print(f"Overall Accuracy: {accuracy_score(y_true_combined, y_pred_combined):.4f}")
    print(classification_report(
        y_true_combined, y_pred_combined,
        target_names=all_class_names,
        zero_division=0,
    ))

    # Confusion matrix
    cm = confusion_matrix(y_true_combined, y_pred_combined)
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title("O-S Model Confusion Matrix (Known + Unknown)")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    tick_marks = np.arange(NUM_KNOWN + 1)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(all_class_names, rotation=45, ha="right")
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(all_class_names)
    fig.colorbar(im)
    fig.tight_layout()
    plt.show()

## 8. Comprehensive Performance Metrics

Detailed evaluation with all standard metrics:
- Per-class Precision, Recall, F1-Score
- False Positive Rate (FPR) per class
- ROC curves and AUC (one-vs-rest)
- Macro and weighted averages
- Open-set specific: Unknown Detection Rate (UDR) and Known Classification Rate (KCR)

In [ ]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_curve, auc,
    roc_auc_score,
    multilabel_confusion_matrix,
)
from sklearn.preprocessing import label_binarize

# --- 8a. Per-class Precision, Recall, F1, and FPR ---
all_class_names = known_class_names + (["Unknown"] if len(X_test_unknown) > 0 else [])
num_all = NUM_KNOWN + (1 if len(X_test_unknown) > 0 else 0)

if len(X_test_unknown) > 0:
    y_true_all = np.concatenate([y_test_known, np.full(len(X_test_unknown), NUM_KNOWN)])
    y_pred_all = np.concatenate([preds_known, preds_unknown])
    probs_all = np.concatenate([probs_known, probs_unknown])
else:
    y_true_all = y_test_known
    y_pred_all = preds_known
    probs_all = probs_known

# Per-class metrics
prec, rec, f1, support = precision_recall_fscore_support(
    y_true_all, y_pred_all, labels=list(range(num_all)), zero_division=0
)

# False Positive Rate per class from multilabel confusion matrix
mcm = multilabel_confusion_matrix(y_true_all, y_pred_all, labels=list(range(num_all)))
fpr_per_class = []
for i in range(num_all):
    tn, fp, fn, tp = mcm[i].ravel()
    fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fpr_per_class.append(fpr_val)

# Build summary table
metrics_df = pd.DataFrame({
    "Class": all_class_names,
    "Precision": prec,
    "Recall": rec,
    "F1-Score": f1,
    "FPR": fpr_per_class,
    "Support": support.astype(int),
})

# Add macro and weighted averages
macro_row = {
    "Class": "Macro Avg",
    "Precision": prec.mean(),
    "Recall": rec.mean(),
    "F1-Score": f1.mean(),
    "FPR": np.mean(fpr_per_class),
    "Support": support.sum(),
}
weighted_row = {
    "Class": "Weighted Avg",
    "Precision": np.average(prec, weights=support),
    "Recall": np.average(rec, weights=support),
    "F1-Score": np.average(f1, weights=support),
    "FPR": np.average(fpr_per_class, weights=support),
    "Support": support.sum(),
}
metrics_df = pd.concat([metrics_df, pd.DataFrame([macro_row, weighted_row])], ignore_index=True)

print("=" * 80)
print("  O-S MODEL — COMPLETE PERFORMANCE METRICS")
print("=" * 80)
print(metrics_df.to_string(index=False, float_format="%.4f"))
print(f"\nOverall Accuracy: {accuracy_score(y_true_all, y_pred_all):.4f}")

In [ ]:
# --- 8b. ROC Curves and AUC (One-vs-Rest) ---
y_true_bin = label_binarize(y_true_all, classes=list(range(num_all)))

# If binary (2 classes), label_binarize returns (N,1) — handle gracefully
if y_true_bin.shape[1] == 1:
    y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Per-class ROC curves ---
ax = axes[0]
auc_scores = {}
colors = plt.cm.tab10(np.linspace(0, 1, num_all))

for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    auc_i = auc(fpr_i, tpr_i)
    auc_scores[all_class_names[i]] = auc_i
    ax.plot(fpr_i, tpr_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} (AUC={auc_i:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — One-vs-Rest")
ax.legend(loc="lower right", fontsize=8)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

# --- Right: Macro-average ROC ---
ax = axes[1]

# Compute macro-average ROC
all_fpr = np.linspace(0, 1, 200)
mean_tpr = np.zeros_like(all_fpr)
valid_classes = 0

for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
    valid_classes += 1

mean_tpr /= max(valid_classes, 1)
macro_auc = auc(all_fpr, mean_tpr)

ax.plot(all_fpr, mean_tpr, color="navy", lw=2,
        label=f"Macro-Average ROC (AUC={macro_auc:.4f})")
ax.fill_between(all_fpr, 0, mean_tpr, alpha=0.1, color="navy")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Macro-Average ROC Curve")
ax.legend(loc="lower right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

fig.suptitle("O-S Model — ROC/AUC Analysis", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(model_dir, "roc_curves.png"), dpi=150)
plt.show()

print("\nPer-class AUC scores:")
for cls_name, auc_val in auc_scores.items():
    print(f"  {cls_name}: {auc_val:.4f}")
print(f"  Macro-Average AUC: {macro_auc:.4f}")

In [ ]:
# --- 8c. Precision-Recall Curves ---
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(10, 6))
ap_scores = {}

for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    prec_i, rec_i, _ = precision_recall_curve(y_true_bin[:, i], probs_all[:, i])
    ap_i = average_precision_score(y_true_bin[:, i], probs_all[:, i])
    ap_scores[all_class_names[i]] = ap_i
    ax.plot(rec_i, prec_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} (AP={ap_i:.4f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — One-vs-Rest")
ax.legend(loc="lower left", fontsize=8)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(model_dir, "precision_recall_curves.png"), dpi=150)
plt.show()

print("\nAverage Precision (AP) per class:")
for cls_name, ap_val in ap_scores.items():
    print(f"  {cls_name}: {ap_val:.4f}")
print(f"  Mean AP (mAP): {np.mean(list(ap_scores.values())):.4f}")

In [ ]:
# --- 8d. Open-Set Specific Metrics + Save All Metrics ---

print("=" * 80)
print("  OPEN-SET RECOGNITION METRICS")
print("=" * 80)

# Known Classification Rate (KCR): accuracy on known test samples
kcr = accuracy_score(y_test_known, preds_known[preds_known != NUM_KNOWN]) if np.sum(preds_known != NUM_KNOWN) > 0 else 0
# More useful: what fraction of known samples are correctly classified (not rejected)
known_acc = np.mean((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_rejection = np.mean(preds_known == NUM_KNOWN)

print(f"Known Classification Accuracy (KCA): {known_acc:.4f}")
print(f"Known Rejection Rate (false unknowns): {known_rejection:.4f}")

if len(X_test_unknown) > 0:
    # Unknown Detection Rate (UDR): fraction of unknown samples detected
    udr = np.mean(preds_unknown == NUM_KNOWN)
    # False Known Rate: unknowns misclassified as known
    fkr = 1.0 - udr
    print(f"Unknown Detection Rate (UDR):        {udr:.4f}")
    print(f"False Known Rate (missed unknowns):   {fkr:.4f}")

    # Harmonic mean of KCA and UDR (balanced open-set score)
    if (known_acc + udr) > 0:
        h_score = 2 * known_acc * udr / (known_acc + udr)
    else:
        h_score = 0.0
    print(f"H-Score (harmonic mean KCA & UDR):    {h_score:.4f}")

# --- Save all metrics to CSV ---
metrics_df.to_csv(os.path.join(model_dir, "performance_metrics.csv"), index=False)

# Save AUC and AP scores
auc_ap_df = pd.DataFrame({
    "Class": list(auc_scores.keys()),
    "AUC": list(auc_scores.values()),
    "AP": [ap_scores.get(c, np.nan) for c in auc_scores.keys()],
})
auc_ap_df.to_csv(os.path.join(model_dir, "auc_ap_scores.csv"), index=False)

print(f"\nMetrics saved to {model_dir}/performance_metrics.csv")
print(f"AUC/AP saved to {model_dir}/auc_ap_scores.csv")

## 9. Save Models & Artifacts

In [ ]:
MODEL_FAMILY = "infogan"
CONFIG_NAME = f"infogan_os_k{NUM_KNOWN}_e{EPOCHS}"
model_dir = nu.get_model_dir(MODEL_FAMILY, CONFIG_NAME)

# Save Keras models
generator.save(os.path.join(model_dir, "generator.keras"))
discriminator.save(os.path.join(model_dir, "discriminator.keras"))
classifier.save(os.path.join(model_dir, "classifier.keras"))
backbone.save(os.path.join(model_dir, "backbone.keras"))

# Save OpenMax artifacts
openmax_data = {
    "mavs": {k: v.tolist() for k, v in mavs.items()},
    "weibull_params": {k: list(v) if v else None for k, v in weibull_params.items()},
    "label_mapping": {str(k): int(v) for k, v in mapping.items()},
    "known_classes": KNOWN_CLASSES,
    "unknown_classes": UNKNOWN_CLASSES,
    "tail_size": TAIL_SIZE,
    "alpha_rank": ALPHA_RANK,
    "distance_type": DISTANCE_TYPE,
    "num_known": NUM_KNOWN,
}
with open(os.path.join(model_dir, "openmax_params.json"), "w") as f:
    json.dump(openmax_data, f, indent=2)

# Save MAVs as numpy (for fast loading)
np.savez(
    os.path.join(model_dir, "openmax_mavs.npz"),
    **{f"class_{k}": v for k, v in mavs.items()},
)

# Save training history
pd.DataFrame(history).to_csv(os.path.join(model_dir, "training_history.csv"), index=False)

# Save metadata
meta = {
    "model_type": "infogan_openmax_os",
    "noise_dim": NOISE_DIM,
    "num_known_classes": NUM_KNOWN,
    "known_classes": KNOWN_CLASSES,
    "unknown_classes": UNKNOWN_CLASSES,
    "known_class_names": known_class_names,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "lambda_info": LAMBDA_INFO,
    "tail_size": TAIL_SIZE,
    "alpha_rank": ALPHA_RANK,
    "distance_type": DISTANCE_TYPE,
    "num_features": len(feature_cols),
    "training_samples": X_train.shape[0],
    "final_d_loss": history["d_loss"][-1],
    "final_g_loss": history["g_loss"][-1],
    "final_q_loss": history["q_loss"][-1],
}
with open(os.path.join(model_dir, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

# Save loss plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["d_loss"]); axes[0].set_title("D Loss")
axes[1].plot(history["g_loss"]); axes[1].set_title("G Loss")
axes[2].plot(history["q_loss"]); axes[2].set_title("Q Loss")
for ax in axes:
    ax.set_xlabel("Epoch")
fig.suptitle(f"InfoGAN Training — {CONFIG_NAME}")
fig.tight_layout()
fig.savefig(os.path.join(model_dir, "training_losses.png"), dpi=150)
plt.close(fig)

print(f"\nAll artifacts saved → {model_dir}")
print(f"Files:")
for f in sorted(os.listdir(model_dir)):
    print(f"  {f}")

## 10. Visualize Generated Samples

Generate fake traffic "images" from the Generator to verify training quality.

In [ ]:
# Generate samples for each class
n_per_class = 4
fig, axes = plt.subplots(NUM_KNOWN, n_per_class, figsize=(n_per_class * 2, NUM_KNOWN * 2))

for cls in range(NUM_KNOWN):
    z = tf.random.normal(shape=(n_per_class, NOISE_DIM))
    c = tf.one_hot(tf.fill([n_per_class], cls), NUM_KNOWN)
    fake_images = generator([z, c], training=False).numpy()

    for j in range(n_per_class):
        ax = axes[cls, j] if NUM_KNOWN > 1 else axes[j]
        ax.imshow(fake_images[j, :, :, 0], cmap="viridis", vmin=-1, vmax=1)
        ax.axis("off")
        if j == 0:
            ax.set_ylabel(known_class_names[cls], fontsize=10, rotation=0, labelpad=60)

fig.suptitle("Generated Traffic Images by Class", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(model_dir, "generated_samples.png"), dpi=150)
plt.show()

## 11. Summary

The O-S (OpenMax-SoftMax) model is now trained and saved. Key outputs:

- **Generator**: can generate synthetic traffic for data augmentation
- **Classifier Q**: classifies known traffic types (SoftMax)
- **OpenMax layer**: detects unknown/zero-day attacks via Weibull tail fitting

**To use the model for inference:**
1. Preprocess input → 69 features → scale → zero-pad → 11×11×1
2. Get activation vectors from classifier Q's penultimate layer
3. Apply OpenMax recalibration
4. If argmax == NUM_KNOWN → unknown attack (zero-day)
5. If argmax < NUM_KNOWN → classified as that known attack type